<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_pipe_startup_new.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 — The Pump Starts

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L9.1 · Laminar Flow**

A liquid cold plate carries a serpentine of copper tube, and a pump drives a
water–glycol coolant through it. When the pump starts, the coolant in the
tube is at rest; it is pushed by the pressure, held back by the wall, and in a
few seconds settles into the velocity profile that every pipe-flow formula
assumes - Hagen–Poiseuille's parabola, from which Darcy's friction factor
$f = 64/\mathrm{Re}$ follows. How does the velocity across the tube grow, and
how long until the flow is steady? The start-up has an **exact solution**, so
finite differences and a physics-informed network are scored against the
truth. Then the coolant's **viscosity** - the number that is hardest to
know, because it changes threefold with temperature and with the glycol in
the mix - is found from a flow meter's record taken before the flow had
settled: by the classical methods, and by a physics-informed network, which
is one way among several and not the only one.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics: Newton's law in a tube, the parabola, Darcy, the start-up | — |
| **4** | the exact solution, and finite differences across the tube for an agreed 1e-03 | — |
| **5** | a PINN with the wall and the start built in, and the comparison | the trial function, the residual |
| **6** | the inverse problem: the coolant's viscosity from a flow meter | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About a minute and a half on a CPU, most of it the network's two trainings. Numbers
are printed with at most two decimals, errors as powers of ten.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'pipe.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "pipe"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "pipe.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import pipe as pp
import numpy as np, torch, matplotlib.pyplot as plt

cc.OUTPUT_DIR = "Ex09.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def online(fn, reps=5):                                    # the same, with the GPU made to finish before the clock stops
    def run():
        fn()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return middle(run, reps)

---

## 1 · The problem

A cold plate under a power module: an aluminium plate, 250 × 130 × 12 mm,
with a copper tube pressed into it that runs back and forth in a
**serpentine** - four straight legs joined by three U-bends. A pump drives
0.3 L/min of coolant through the tube. The drawing below shows it; this
notebook solves the flow in **one straight leg**, in the part far enough from
the bend before it (orange). The bends are the subject of mini project
MP9.1A.

At $t = 0$ the pump starts: a pressure drop of $G$ pascal per metre is
applied along the tube, and held. Wanted: the velocity of the coolant at
every point of the tube's cross-section, over the first four seconds; how
long before it is steady; and the pressure drop once it is.

## 2 · The data

| | |
|---|---|
| plate | aluminium, 250 × 130 × 12 mm; four legs of 230 mm, three U-bends of 12 mm radius |
| tube | copper, 6 mm inside diameter ($R$ = 3 mm) |
| coolant | 50/50 water–ethylene glycol at 40 °C: density $\rho$ = 1065 kg/m³, viscosity $\mu$ = 2.5 mPa·s |
| flow | 0.3 L/min, a mean speed of 0.18 m/s |
| start | the coolant at rest; the pressure drop $G$ applied at $t = 0$ and held |
| window | 0 to 4 s |

**These are typical values, typed from memory, not measurements**: check
them before you quote a result. The pressure drop is taken to be held from
$t = 0$: a pump settles in a small fraction of the seconds the coolant takes
to follow it.

In [ ]:
pp.describe_problem()
pp.draw_domain(); plt.show()                                  # the plate, the serpentine, the solved leg, the section A-A

**What you should see.** The data, and two drawings. On the left the plate
in 3-D: the tube's four legs in blue, the bends in purple, the module on top
in red, and in orange the part of the second leg this notebook solves - the
last 94 mm of it, beyond the 136 mm the flow needs to recover from the bend
before it (section 3 says why). On the right the tube's cross-section at
A-A, with what is known on it.

---

## 3 · The physics

**A parcel** is a small lump of the liquid - small enough that its velocity
is one number, large enough to hold many molecules - followed as it moves.
L9.1's *Navier–Stokes* slide writes Newton's law for it: mass per volume
times acceleration equals the forces per volume,

$$\rho\big(\mathbf{u}_t + (\mathbf{u}\cdot\nabla)\mathbf{u}\big) = -\nabla p + \mu\,\nabla^2\mathbf{u}, \qquad \nabla\cdot\mathbf{u} = 0$$

with $\mathbf{u}$ the velocity (a vector), $p$ the pressure and $\mu$ the
viscosity. The acceleration has two parts: $\mathbf{u}_t$, the velocity
changing in time at a fixed point, and $(\mathbf{u}\cdot\nabla)\mathbf{u}$,
the velocity changing because the parcel moves to where the velocity is
different.

**In the tube.** Take $x$ along the tube's axis and $r$ the distance from
it. The velocity has three components: $u$ along the tube, $v$ towards the
wall, $w$ round the axis. Three facts reduce them to one:

1. **Round and straight**: nothing distinguishes one angle from another, so
   nothing makes the liquid swirl: $w = 0$, and nothing depends on the angle.
2. **Far from the bends** (beyond the *entrance length*, below): the profile
   no longer changes along the tube, $\partial u/\partial x = 0$. This is
   called **fully developed** flow.
3. **The volume is kept**, $\nabla\cdot\mathbf{u} = 0$, which in a tube reads
   $\partial u/\partial x + \frac{1}{r}\,\partial (r v)/\partial r = 0$.
   With fact 2 the first term is zero, so $r v$ is the same at every radius;
   on the wall $v = 0$, so $v = 0$ everywhere.

What is left is $u(r, t)$: **the velocity points along the tube and changes
only across it**. Then the acceleration's second part vanishes,

$$(\mathbf{u}\cdot\nabla)u = u\,\frac{\partial u}{\partial x} + v\,\frac{\partial u}{\partial r} = u \cdot 0 + 0 \cdot \frac{\partial u}{\partial r} = 0$$

In words: a parcel moves along the tube at its own radius, where the speed is
the same at every point along, so moving never speeds it up or slows it
down. The pressure can only vary along the tube, and it falls by $G$ per
metre. Newton's law along the tube becomes

$$\rho\,\frac{\partial u}{\partial t} = G + \mu\,\frac{1}{r}\frac{\partial}{\partial r}\Big(r\,\frac{\partial u}{\partial r}\Big)$$

the push of the pressure, against the drag of the neighbours - the last term
is the Laplacian $\nabla^2 u$ written for a round tube.

**The conditions** that make the answer unique:

| where | condition | why |
|---|---|---|
| wall, $r = R$ | $u = 0$ | the liquid sticks to the wall (*no slip*) |
| axis, $r = 0$ | $\partial u/\partial r = 0$ | the profile is the same on both sides of the axis |
| start, $t = 0$ | $u = 0$ | the coolant is at rest when the pump starts |
| along the tube | $G$ = 393 Pa/m, from $t = 0$ | the pump's push |

**Why this one can be solved by hand.** The three facts, plus a liquid whose
viscosity does not depend on the speed (Newtonian) and a flow that stays
laminar, remove the one nonlinear term of Navier–Stokes and every coordinate
but $r$. What is left is **linear**, in one space coordinate and time. Bend
the tube, or let it turn turbulent, and none of this holds - which is why
MP9.1A needs a solver.

**Steady: Hagen–Poiseuille.** Once nothing changes, $\partial u/\partial t =
0$; integrate twice in $r$, with the two conditions on the wall and the axis:

$$u(r) = \frac{G R^2}{4\mu}\Big(1 - \frac{r^2}{R^2}\Big)$$

a **parabola**, fastest on the axis and zero on the wall (Hagen 1839,
Poiseuille 1840; White, *Fluid Mechanics*, ch. 6). It has no $t$ because it
is the state that has stopped changing. Its **mean speed**, the flow rate
over the area, is

$$\bar u = \frac{1}{\pi R^2}\int_0^R u\;2\pi r\,dr = \frac{G R^2}{8\mu}$$

which is **Darcy's law** of L9.1's *Inertia and Darcy* slide,
$\bar u = (K/\mu)\,G$, with the permeability of a round tube $K = R^2/8$.
Here it is exact, not an approximation: the inertia was not neglected, it is
zero. Written as a friction factor it is $f = 64/\mathrm{Re}$, with
$\mathrm{Re} = \rho\bar u D/\mu$ the Reynolds number.

**In time: the start-up.** The same equation, with $\partial u/\partial t$
kept, from the coolant at rest. Its solution is also known exactly
(Szymański 1932; White, *Viscous Fluid Flow*, ch. 3): **the parabola, less
the part that has not yet arrived**,

$$u(r, t) = \frac{G R^2}{4\mu}\Big[\Big(1 - \frac{r^2}{R^2}\Big) - \sum_{n=1}^{\infty} \frac{8\,J_0(\lambda_n r/R)}{\lambda_n^3\,J_1(\lambda_n)}\,e^{-\lambda_n^2 \nu t/R^2}\Big]$$

$J_0$ and $J_1$ are Bessel functions - the round tube's version of a cosine
and a sine - and $\lambda_n$ = 2.405, 5.520, 8.654, … are the zeros of $J_0$.
Each term dies away at its own rate; the slowest within about
$R^2/\nu$ = 3.8 s, with $\nu = \mu/\rho$. It is not a second answer to a
second problem: it is the answer for every $t$, and Hagen–Poiseuille is what
it becomes when $t$ is large. **The mean speed in time** follows by the same
integral, term by term:

$$\bar u(t) = \frac{G R^2}{4\mu}\Big[\frac{1}{2} - \sum_{n=1}^{\infty}\frac{16}{\lambda_n^4}\,e^{-\lambda_n^2 \nu t/R^2}\Big]$$

**Scaled units.** Every variable is divided by a scale of its own, so the
numbers are of size one; a hat marks a scaled variable:

| | scaled | scale |
|---|---|---|
| velocity | $\hat u = u/u_c$ | $u_c = GR^2/4\mu$ = 0.354 m/s, the steady centre speed |
| time | $\hat t = t/T$ | $T$ = 4 s, the window |
| radius | $s = (r/R)^2$ | from 0 on the axis to 1 on the wall |

$s$ is used instead of $r/R$ because then the axis needs no special care:
any function of $s$ is the same on both sides of the axis. By the chain rule
the equation becomes

$$\frac{\partial \hat u}{\partial \hat t} = a + 4\kappa\,\big(s\,\hat u_{ss} + \hat u_s\big), \qquad \kappa = \frac{\nu T}{R^2} = 1.04, \qquad a = 4\kappa$$

with two numbers left: $\kappa$, the **viscosity** in these units (in the
code `pp.C_SCALED`), and $a$, the **pressure's push** (`pp.A_SCALED`). In
scaled units the mean speed is simply $\int_0^1 \hat u\,ds$, because $s$ is
uniform in the area of the section.

**Two checks before anything is solved.** $\mathrm{Re} \approx 450$, far
below the 2300 at which pipe flow turns turbulent. And after each bend the
parabola needs about $0.05\,\mathrm{Re}\,D \approx$ 136 mm to re-form - the
**entrance length** - so only the last 94 mm of each 230 mm leg is fully
developed; that is the orange part in the drawing. The cell below draws the
exact start-up.

In [ ]:
s = np.linspace(0, 1, 201); r_mm = 1e3 * pp.R_TUBE * np.sqrt(s)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for t_s in (0.1, 0.3, 0.6, 1.0, 2.0, 4.0):
    u = pp.V * pp.exact(s, t_s / pp.T_END)                     # the exact velocity across the tube, m/s
    ax[0].plot(np.r_[-r_mm[::-1], r_mm], np.r_[u[::-1], u], label=f"{t_s:g} s")
ax[0].plot(np.r_[-r_mm[::-1], r_mm], pp.V * np.r_[1 - s[::-1], 1 - s], "k--", lw=1, label="the parabola")
ax[0].set_xlabel("r  [mm], across the tube"); ax[0].set_ylabel("u  [m/s]"); ax[0].legend(frameon=False, fontsize=8, ncol=2)
ax[0].set_title("the velocity across the tube at six instants", fontsize=10)
t = np.linspace(0, 1, 201)
ax[1].plot(t * pp.T_END, pp.V * pp.exact_mean(t), "k", lw=2, label="mean speed: flow rate / area")
ax[1].plot(t * pp.T_END, pp.V * pp.exact(0.0, t), color="tab:blue", lw=1.5, label="speed on the axis")
ax[1].axhline(pp.U_MEAN, color="0.5", ls="--", lw=1, label="steady mean (Darcy)")
ax[1].set_xlabel("t  [s]"); ax[1].set_ylabel("speed  [m/s]"); ax[1].legend(frameon=False, fontsize=8)
ax[1].set_title("the mean speed and the speed on the axis in time", fontsize=10)
plt.tight_layout(); plt.show()

**What you should see.** On the left the profile at six instants: **flat in
the middle and steep at the wall** at first - the whole core is pushed at
once, and only the liquid near the wall feels the drag - then rounding into
the parabola. On the right the mean speed (what a flow meter reads: the flow
rate over the area) and the speed on the axis, half-way to steady after
about half a second and within a per cent after about three.

---

## 4 · The reference, and finite differences

**The reference** is the exact solution of section 3, `pp.exact(s, t)`:
the series summed to 400 terms, which is exact to rounding at every instant
of the window.

**Finite differences.** Nodes from the axis to the wall, $h$ apart - each
node stands for a ring of the section, because $u$ is the same all round it.
At each node the derivatives are replaced by differences between neighbours,
$u_{rr} \approx (u_{j+1} - 2u_j + u_{j-1})/h^2$ and
$u_r \approx (u_{j+1} - u_{j-1})/2h$; on the axis, where $u_r/r$ becomes
$u_{rr}$, the Laplacian is $2u_{rr}$, with the node beyond the axis the
mirror of the one before it; on the wall $u = 0$. In time, Crank–Nicolson:
the average of the old and the new right-hand side, so one sparse solve per
step with the matrix factorised once.

**The physics-informed network** of section 5 has no grid: it is asked the
equation at points scattered over radius *and* time at once.

**The agreed accuracy is 1e-03** of the steady centre speed - 0.35 mm/s - at
every radius and every instant. The cell below refines the grid and the step
together until it is met, and draws where each method puts its unknowns.

In [ ]:
TS = np.linspace(0, 1, 41)                                    # the 41 instants every method is scored at, 0.1 s apart
TARGET = 1e-3
def fdm_error(n, steps):
    eta, ts, U = pp.fdm(n, steps)                             # nodes from the axis to the wall, Crank-Nicolson
    return np.abs(U - np.array([pp.exact(eta ** 2, t) for t in ts])).max()
for n, steps in ((11, 40), (21, 80), (41, 160), (81, 320)):
    err = fdm_error(n, steps); sec = online(lambda: pp.fdm(n, steps), reps=3)
    print(f"  {n:3d} nodes, {steps:3d} steps: worst error {err:.2e}, one run {1e3 * sec:.1f} ms" + ("   <- meets 1e-03" if err <= TARGET else ""))
N_FDM, STEPS = 21, 80                                         # the coarsest grid that meets it

rng = np.random.default_rng(0)
PTS = np.c_[rng.uniform(0, 1, 3000), rng.uniform(0, 1, 3000) ** 2]   # the network's collocation points (s, t), more early
pp.draw_methods(N_FDM, PTS[:, 0], PTS[:, 1]); plt.show()

**What you should see.** The error falling by about four with every halving -
4.76e-03, 9.08e-04, 2.68e-04, 7.15e-05 - and **21 nodes with 80 steps the
first under 1e-03**, in about a millisecond. Then where the unknowns are: the
plate from above with the section A-A in the solved part of the leg; the
grid's 21 nodes across the section, each a ring (thin lines); and the
network's 3000 points over radius and time - more of them early, where the
profile changes fastest, and uniform in $s$, which is uniform in the area of
the section, so they thin out towards the axis.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(s, \hat t)$ and returns the scaled velocity $\hat u$.
It is four layers of 32 tanh neurons, trained on those 3000 collocation
points.

**The wall and the start are built in.** The trial function

$$\hat u(s, \hat t) = (1 - s)\; \hat t\; \mathcal{N}(s, \hat t)$$

is exactly zero on the wall ($s = 1$) and exactly zero at the start
($\hat t = 0$), whatever the network $\mathcal{N}$ does: **hard
enforcement**, one factor for each condition. Taking $s$ as the input builds
in the third condition, symmetry about the axis, for any weights. Nothing is
left for a loss term but the equation itself.

**TODO 1** is the trial function; **TODO 2** the residual. The equation is
divided by $\kappa$ = 1.04, the scaled viscosity of section 3, so that the
viscous term is of size one: $(\hat u_{\hat t} - a)/\kappa - 4(s\,\hat u_{ss} + \hat u_s)$.

In [ ]:
class Tube(torch.nn.Module):                                  # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # dense tanh network: s and t in
    def forward(self, p):
        s, t = p[:, 0:1], p[:, 1:2]
        # TODO 1: the trial function - (1 - s) times t times the network.
        #
        #   return (1 - s) * t * self.net(p)                      # zero on the wall and at the start
        raise NotImplementedError("Write the trial function")

In [ ]:
# TODO 2: the residual of the scaled equation, divided by c.
#
#   def residual(model, p, c, a):
#       u = model(p)
#       du = grad(u, p); u_s, u_t = du[:, 0:1], du[:, 1:2]        # column 0 is s, column 1 is t
#       u_ss = grad(u_s, p)[:, 0:1]
#       s = p[:, 0:1]
#       return (u_t - a) / c - 4 * (s * u_ss + u_s)
raise NotImplementedError("Write residual(model, p, c, a)")

In [ ]:
pts = to_tensor(PTS, requires_grad=True)                     # the collocation points of section 4, differentiable
model = Tube().to(DEVICE)                                     # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, pts, pp.C_SCALED, pp.A_SCALED).pow(2).mean()   # the loss: the equation, nothing else
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
SG, TG = np.meshgrid(np.linspace(0, 1, 101), TS)              # 101 radii x 41 instants
truth = pp.exact(SG, TG)
grid_in = to_tensor(np.c_[SG.ravel(), TG.ravel()])                # the same radii and instants, as network inputs
def pinn_field():
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(grid_in)).reshape(SG.shape)     # the trained network at every radius and instant
e_pin = np.abs(pinn_field() - truth).max()
eta, ts, U = pp.fdm(N_FDM, STEPS)
e_fdm = np.abs(U - np.array([pp.exact(eta ** 2, t) for t in ts])).max()
t_fdm = online(lambda: pp.fdm(N_FDM, STEPS))                  # one run of the grid, the whole window, on this runtime
t_pin = online(pinn_field)                                    # one evaluation of the trained network, the whole window, on this runtime

# the time column is what runs ONLINE: one grid run against one evaluation of the trained network, on the same runtime.
# The training is offline, paid once, and reported on its own line.
print(f"{'':34s}{'unknowns':>10s}{'worst error':>14s}{'online':>12s}")
print(f"  {'exact series, 400 terms':32s}{'-':>10s}{'-':>14s}{'-':>12s}")
print(f"  {f'finite differences, {N_FDM} x {STEPS}':32s}{N_FDM:10d}{e_fdm:14.2e}{1e3 * t_fdm:9.2f} ms")
print(f"  {'PINN, 4 x 32':32s}{parameter_count(model):10d}{e_pin:14.2e}{1e3 * t_pin:9.2f} ms")
print(f"  the network's training, offline: {train_time:.0f} s" +
      (f", repaid after {train_time / (t_fdm - t_pin):.0f} runs" if t_fdm > t_pin else ", never repaid on time alone"))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, c in ((4, "tab:blue"), (10, "tab:orange"), (40, "tab:green")):
    ax[0].plot(np.sqrt(SG[k]), pp.V * truth[k], color=c, lw=2, label=f"exact, {TS[k] * pp.T_END:.1f} s")
    ax[0].plot(eta, pp.V * U[k], "s", ms=3, color=c)
    ax[0].plot(np.sqrt(SG[k]), pp.V * pinn_field()[k], ":", color="k", lw=1.2)
ax[0].set_xlabel("r / R"); ax[0].set_ylabel("u  [m/s]"); ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title("lines exact, squares the grid, dots the network", fontsize=9)
ax[1].semilogy(TS * pp.T_END, np.abs(U - np.array([pp.exact(eta ** 2, t) for t in ts])).max(axis=1), label="finite differences")
ax[1].semilogy(TS * pp.T_END, np.abs(pinn_field() - truth).max(axis=1), label="PINN")
ax[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 1e-03")
ax[1].set_xlabel("t  [s]"); ax[1].set_ylabel("worst error across the tube"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex09.1_report.png"), dpi=150); plt.show()

def ring(s):
    # the grid node whose ring each radius s falls in: the grid knows u only at its nodes
    return np.abs(np.sqrt(s)[:, None] - eta[None, :]).argmin(axis=1)
def fdm_at(s, t):
    # the grid's velocity in m/s: each node's value over its own ring, at the stored instant nearest to t
    k = int(np.abs(ts - t).argmin())
    return pp.V * U[k][ring(s)]
def fdm_error_map(s, t):
    # the grid's error in m/s, scored at its own nodes and drawn over their rings
    k = int(np.abs(ts - t).argmin()); j = ring(s)
    return pp.V * np.abs(U[k][j] - pp.exact(eta[j] ** 2, ts[k]))
def pinn_at(s, t):
    # the network's velocity in m/s at the radii s and the instant t
    with torch.no_grad():  # no gradient tracking: evaluation only
        return pp.V * to_numpy(model(to_tensor(np.c_[s, np.full_like(s, t)]))).ravel()   # (s, t) -> the trained network -> m/s
exact_at = lambda s, t: pp.V * pp.exact(s, t)                 # the exact velocity in m/s
pp.draw_sections({"exact": exact_at, "finite differences": fdm_at, "PINN": pinn_at}, (0.4, 1.0, 4.0)); plt.show()
pp.draw_sections({"error of finite differences": fdm_error_map,
                  "error of the PINN": lambda s, t: np.abs(pinn_at(s, t) - exact_at(s, t))}, (0.4, 1.0, 4.0)); plt.show()

**What you should see.** Both within the agreed 1e-03 of the exact velocity
at every radius and instant: finite differences 9.08e-04 with 21 unknowns,
the network about 6e-04 with some 3 300 weights. **Online** the trained
network gives the whole window in about 0.4 ms and the grid in about 1 ms;
but the network's half-minute of training is repaid only after some
60 000 runs.

The first figure plots the profiles (lines exact, squares the grid, dots
the network) and the worst error in time; the grid's error is largest early,
when the profile is steep at the wall, and the network's is spread over the
window. Then the **cross-section in colour**, at 0.4 s, 1 s and 4 s: the
exact velocity, the grid's (each node's value over its own ring - the grid
knows nothing between its nodes) and the network's look the same: a flat
core with a thin slow ring at the wall at first, the parabola's bright
centre at the end. The last maps show where each is wrong, in m/s - at most
about 1e-04 m/s or less, under 0.1 % of the centre speed: the grid's largest
in the core during the first second and a few times 1e-06 once the flow is
steady; the network's in rings, largest on the axis at the end.

**By accuracy** the two are close, the network a little ahead; **by time to a
first answer** the grid wins by far, because it needs no training. For a
straight tube with an exact solution there is no reason to prefer the
network - which is why the next section asks it a different question.

---

## 6 · The inverse problem: the coolant's viscosity

The viscosity was the one number in section 2 that is hard to know. A
water–glycol mix is three times as viscous at 20 °C as at 80 °C, and the
glycol fraction drifts as a loop is topped up with water. Suppose the loop
was filled with a stronger mix than specified, and the plate is cooler: the
true viscosity is **3.5 mPa·s**, not 2.5. What is known: the pressure drop
$G$ the pump applies, the density, and a **flow meter**'s record of the mean
speed $\bar u$, every 0.05 s for the **first second** after the pump starts,
with noise of 1 % - before the flow is steady.

**Four ways to find the viscosity.** An inverse problem is not something
only a network can do: any model that predicts the readings can be fitted to
them.

1. **Darcy's law**, the classical engineering way: take the last reading as
   the steady mean speed and solve $\bar u = GR^2/8\mu$ for $\mu$. It needs
   no model of the start-up, and it is only right if the flow has settled.
2. **Least squares on the exact solution**: adjust $\kappa$ until the exact
   mean speed of section 3 matches the readings. It is right, and it is
   possible only because this straight tube has an exact solution.
3. **Least squares on finite differences**: the same, with the grid of
   section 4 as the model - run it, compare its mean speed with the
   readings, adjust $\kappa$, run it again. This is the usual classical way,
   and it needs no exact solution, only a solver cheap enough to run many
   times.
4. **The physics-informed network**: a fresh network for the velocity, and
   one more trainable number, $\kappa = e^{\beta}$, an exponential so that it
   cannot go negative. It is **not an output of the network**: it is a
   number beside the weights, in the residual, changed by the optimiser with
   them. The loss is the residual plus the misfit between the network's mean
   speed, $\int_0^1 \hat u\,ds$, and the readings. $\kappa$ starts at the
   nominal 1.04, 29 % low.

In [ ]:
from scipy.optimize import least_squares
MU_TRUE = 3.5e-3                                              # Pa s: the coolant actually in the loop - unknown to the methods
c_true = MU_TRUE / pp.RHO * pp.T_END / pp.R_TUBE ** 2
T_REC = 0.25                                                  # the record: the first second (a quarter of the window)
t_m, q_m = pp.flow_meter(c_true, t_end_record=T_REC, every=T_REC / 20)   # 20 readings of the mean speed, scaled, with noise
to_mu = lambda c: c * pp.RHO * pp.R_TUBE ** 2 / pp.T_END      # the scaled viscosity kappa -> Pa s

mu_darcy = pp.G * pp.R_TUBE ** 2 / (8 * q_m[-1] * pp.V)       # 1. Darcy: the last reading taken as steady
fit = least_squares(lambda p: pp.exact_mean(t_m, c=p[0]) - q_m, [pp.C_SCALED])   # 2. the exact series fitted
mu_lsq = to_mu(fit.x[0])

def grid_mean(c):
    # the mean speed of the section-4 grid at the 20 reading instants, for a trial scaled viscosity c
    eta_g, ts_g, U_g = pp.fdm(N_FDM, STEPS, keep=STEPS, c=c)  # stored every step: the readings fall on steps
    return pp.fdm_mean(eta_g, U_g)[1:len(t_m) + 1]
t0 = time.perf_counter()
fit_g = least_squares(lambda p: grid_mean(p[0]) - q_m, [pp.C_SCALED])   # 3. the grid inside the optimiser
t_grid_fit = time.perf_counter() - t0
mu_grid = to_mu(fit_g.x[0])

gl_x, gl_w = np.polynomial.legendre.leggauss(16)              # 16 Gauss points across the section, for the mean speed
SQ = to_tensor(np.c_[np.tile(0.5 * (gl_x + 1), len(t_m)), np.repeat(t_m, 16)])   # (s, t) at every reading
WQ = to_tensor(0.5 * gl_w).reshape(-1)                        # their weights
QM = to_tensor(q_m).reshape(-1)                               # the readings
pts_rec = to_tensor(np.c_[rng.uniform(0, 1, 3000), T_REC * rng.uniform(0, 1, 3000) ** 2], requires_grad=True)   # points over the record only

class Inverse(torch.nn.Module):                               # the tube's network and one more trainable number
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # a fresh dense tanh network: s and t in
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(pp.C_SCALED))))   # kappa = exp(beta), starts at the nominal
    def forward(self, p):
        s, t = p[:, 0:1], p[:, 1:2]
        return (1 - s) * t * self.net(torch.cat([s, t / T_REC], 1))   # the same trial function, time scaled to the record
inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 100.0                                                   # the weight of the readings
def inverse_loss():
    c = torch.exp(inv.beta)                                   # the viscosity being identified, always positive
    mean = (inv(SQ).reshape(len(t_m), 16) * WQ).sum(dim=1)    # the network's mean speed at each reading
    return residual(inv, pts_rec, c, pp.A_SCALED).pow(2).mean() + W_D * (mean - QM).pow(2).mean()
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
mu_pinn = to_mu(float(torch.exp(inv.beta)))
for name, mu in (("Darcy, last reading as steady", mu_darcy), ("least squares, exact series", mu_lsq),
                 ("least squares, finite differences", mu_grid), ("physics-informed network", mu_pinn)):
    print(f"  {name:34s}{1e3 * mu:6.2f} mPa s   ({100 * (mu / MU_TRUE - 1):+.1f} %)")
print(f"  true {1e3 * MU_TRUE:.2f} mPa s; the grid fit took {fit_g.nfev} runs and {1e3 * t_grid_fit:.0f} ms, the network {inv_time:.0f} s of training")

**What you should see.** Darcy returns about 4.0 mPa·s, **15 % high**: after
one second the flow is not yet steady, the last reading is low, and a low
speed under a known pressure reads as a viscous liquid. The other three all
return 3.53 to 3.56 mPa·s, within 2 % - the error is the noise of the
readings, not the method. **The grid inside the optimiser is the cheapest**:
a handful of runs, some ten milliseconds, against the network's forty seconds
of training. For this tube the network brings nothing the classical methods
lack. What it would bring is elsewhere: it finds the field and the number in
one training, with no solver run again and again, and it needs no solver at
all - which matters when the solver is slow or does not exist, as in the
bend of MP9.1A.

---

## 7 · What the notebook says

* **Darcy's 64/Re is the steady limit of Newton's law in a tube.** The
  parabola gives it in two lines; it holds once the flow has settled and the
  entrance length after each bend has passed.
* **The start-up is exact, so the grid and the network fought on the truth.**
  Both met the agreed 1e-03; the grid did it with 21 unknowns in a
  millisecond and no training, the network a little more exactly after half
  a minute of training.
* **Three conditions were built into the network**: the wall and the start by
  factors, the symmetry about the axis by the input $s$. The loss was the
  equation alone.
* **The viscosity is the number worth finding.** It changes with temperature
  and with the mix, and it sets the pressure drop and the pump's work.
* **An inverse problem is not a network's privilege.** Any model that
  predicts the readings can be fitted to them: the exact series, the grid in
  an optimiser and the network all found the viscosity within 2 %, the grid
  fastest. A classical formula used outside its conditions was the worst
  answer: Darcy on a flow that had not settled was 15 % off. The network's
  way pays where no cheap solver exists - a bend, which is what MP9.1A asks.
* **None of the numbers is a measured coolant.** The properties are typical
  values from memory; check them against the supplier's data before
  designing with them.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex09.1_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L9.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Which terms of Newton's law for a liquid are left in a straight tube far from the bends, and why is the convective acceleration zero there? (-> L9.1 Q1)": """
""",
    "What is the Reynolds number of this flow, what does it tell you about laminar flow, and how long after a bend before the parabola - and Darcy's 64/Re - apply? (-> L9.1 Q2)": """
""",
    "Which three conditions did the trial function and the input s build into the network, and why was the residual divided by kappa? (-> L9.1 Q3)": """
""",
    "Why was Darcy's viscosity 15 % off while the three fitted models were within 2 %, which of those three was cheapest, and what would make the network the right choice instead? (-> L9.1 Q4)": """
""",
    "To conclude: for this straight tube the grid won on time and matched on accuracy. Name the situation in this cold plate where you would use the network instead, and say what you would measure to check it. (-> L9.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | unknowns | worst error | online |", "|---|---|---|---|",
            f"| finite differences, {N_FDM} x {STEPS} | {N_FDM} | {e_fdm:.2e} | {1e3 * t_fdm:.2f} ms |",
            f"| PINN, 4 x 32 | {parameter_count(model)} | {e_pin:.2e} | {1e3 * t_pin:.2f} ms (trained offline in {train_time:.0f} s) |",
            "", "| viscosity from the first second | mPa s | off by |", "|---|---|---|",
            f"| Darcy, last reading as steady | {1e3 * mu_darcy:.2f} | {100 * (mu_darcy / MU_TRUE - 1):+.1f} % |",
            f"| least squares, exact series | {1e3 * mu_lsq:.2f} | {100 * (mu_lsq / MU_TRUE - 1):+.1f} % |",
            f"| least squares, finite differences ({fit_g.nfev} runs, {1e3 * t_grid_fit:.0f} ms) | {1e3 * mu_grid:.2f} | {100 * (mu_grid / MU_TRUE - 1):+.1f} % |",
            f"| physics-informed network | {1e3 * mu_pinn:.2f} | {100 * (mu_pinn / MU_TRUE - 1):+.1f} % |",
            "", f"True viscosity {1e3 * MU_TRUE:.2f} mPa s. The coolant's properties are typical values, not measurements."]
    out = ["# Ex_09.1 — The Pump Starts", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex09.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex09.1_report.md into Ex09.1_report.pdf, with any figure
# saved as Ex09.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex09.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex09.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP9.1A · The serpentine

**The problem.** The notebook solved one straight leg, far from the bends. The coolant meets
four legs and three U-bends of 12 mm radius, and the speed across the tube is
not the same from section to section: in a leg it is the notebook's parabola;
in a bend the fast liquid in the middle is thrown outwards and comes back along
the walls, so the fastest liquid moves towards the outer wall and two
counter-rotating vortices appear beside the axial flow (Dean vortices); and
the bends raise the pressure the pump must supply. How does the speed
distribution change from a leg to a bend, and what does the whole serpentine
cost in pressure?

| | |
|---|---|
| tube, coolant, flow | as in the notebook: 6 mm, water–glycol at 40 °C, 0.3 L/min, Re = 452 |
| serpentine | four legs of 230 mm, three U-bends of 12 mm radius to the tube's axis: $a/R_c$ = 0.25, Dean number $\mathrm{Re}\sqrt{a/R_c}$ = 226 |
| asked | the speed across a leg and across a bend, in colour; the friction in a bend; the pressure drop of the serpentine |

The bend radius is a typical value, sourced before the data is released.

**Why a plain PINN is not enough.** In a bend the flow is three-dimensional: an axial velocity and a secondary flow
across the section, coupled - the secondary flow moves the axial velocity, and
the axial velocity drives the secondary flow through the centrifugal force. Two
outputs, fourth derivatives through a stream function, and walls where both the
velocity and its slope vanish. No series gives it.

**What you build.** A PINN for the fully developed flow across a bend: the axial velocity and the
secondary stream function as two outputs, in Dean's form of the equations, the
no-slip wall built in. Then the serpentine assembled from the notebook's legs and
your bends: the speed maps section by section, and the total pressure drop.

**The ground truth you get.** `MP9.1A_new_truth.npz`, from `tools/miniprojects/pipe_truth.py`: the bend by finite
volumes on a polar grid with Newton's method (`dean.py`), 48 × 96 cells, checked
against Dean's own series at small Dean numbers (0.99623 against 0.99649 at
$C$ = 100) and by refining the grid; the legs by the notebook's exact parabola.
**Not in the ground truth**: the flow just after a bend, where the bend's profile
turns back into the parabola over the entrance length - no solver here computes it,
and the pressure drop below leaves it out.

**Worked example.** 

| across | fastest liquid | where | friction over 64/Re | pressure drop |
|---|---|---|---|---|
| a leg (the notebook) | 0.354 m/s | on the axis | 1 | 393 Pa/m |
| a bend, 24 × 48 cells | 0.311 m/s | 0.73 R towards the outer wall | 2.014 | 791 Pa/m |
| a bend, 48 × 96 cells | 0.309 m/s | 0.74 R towards the outer wall | 1.995 | 784 Pa/m |

In a bend the fastest liquid moves three quarters of the way to the outer wall and the
friction doubles (Mori and Nakayama's correlation, quoted from memory: 2.02). The
serpentine: legs 362 Pa plus bends 89 Pa, **450 Pa**, against 406 Pa for a straight tube of
the same length - before the entrance length after each bend is counted.

**Requirements.**

* The friction in a bend within **3 %** of the ground truth's.
* The axial velocity across a bend within **2 %** of the straight leg's centre speed everywhere.
* The speed maps of a leg and of a bend side by side, the two vortices drawn, and the serpentine's pressure drop within **3 %** of the ground truth's sum.

**What you hand in.** The speed across a leg and across a bend against the ground truth, the vortices, the friction in the bend, the serpentine's pressure drop, and a paragraph on what the bends cost the pump and what the missing entrance lengths would add.

### MP9.1B · The pump's speed reference

**The problem.** In a cold plate the coolant's speed is what a controller can change: the power
module's power steps up and down with the load, and the pump must be told how
fast to run so that the module never exceeds its limit - but no faster, because
pumping costs energy and wears the pump. Given the module's power over the next
half hour, a staircase, what flow should be sent to the pump at every instant?

| | |
|---|---|
| plate | the notebook's cold plate as one thermal mass, 948 J/K; the tube 1.03 m long inside it |
| module | 0.05 K/W from its junction to the plate; limit 80 °C |
| coolant | the notebook's, supplied at 25 °C; $c_p$ = 3400 J/(kg K), $k$ = 0.40 W/(m K); Prandtl number 21 |
| pump | 0.05 to 1.5 L/min, laminar throughout |
| power | 120 W for 20 minutes, then 300 W for 400 s, then 150 W for 400 s |

All values typical, from memory, sourced before the data is released.

**Why a plain PINN is not enough.** Two models meet: the flow sets the heat transfer, and the heat transfer sets the
temperature that the flow must keep down. With a Prandtl number of 21 the
temperature profile in the tube needs 2.9 m to develop and the tube is 1 m, so
the heat transfer coefficient is the developing one, which grows with the flow -
the fully developed laminar value would say the speed does not matter at all.
And the answer is a whole function of time that must look ahead: a controller
that sees only the present power cannot cool the plate before a step.

**What you build.** A network that outputs the flow reference as a function of time, bounded between
0.05 and 1.5 L/min by its last layer, with the plate's energy balance as the
physics: the plate's temperature is a second network (or integrated), the module's
limit a penalty, the pumping power - laminar, so the pressure drop is proportional
to the flow and the power to its square - the cost.

**The ground truth you get.** `MP9.1B_new_truth.npz`, from `tools/miniprojects/pipe_truth.py`: the least-pumping flow
by direct transcription - the flow held on 20 s intervals, the plate's energy
balance integrated exactly between them, minimised by SLSQP, checked on 40 s
intervals - and the heat transfer by Sieder and Tate's correlation.

**Worked example.** Started cold under 120 W, the plate is cooled at the least flow, 0.05 L/min, for
the first 700 s. **Then, eight minutes before the step, the flow starts to rise** -
0.10 L/min at 1000 s, 0.17 at the step - so that the plate is down to 55 °C when the
power reaches 300 W and the module's limit allows the plate only 65 °C. Under 300 W
it runs at 0.32 L/min; under 150 W it falls back to 0.05 - 0.07. The module reaches
exactly 80.00 °C and never passes it; pumping 0.98 J (the same on 40 s and 20 s
intervals). A controller that sees only the present power stays at 0.05 L/min
until the step, finds the plate too warm, runs at the full 1.5 L/min and **still
reaches 86.47 °C**. The flow helps through the developing heat transfer: the
coolant takes 2.46 W/K at 0.05 L/min, 7.23 at 0.3 and 14.48 at 1.5.

**Requirements.**

* The module never above **80.2 °C**.
* The pumping within **5 %** of the ground truth's.
* The flow before the step up raised in time, and a paragraph on why a controller that sees only the present power cannot do this.

**What you hand in.** The flow reference and the module's temperature against the ground truth, the pumping energy, the reactive controller's overshoot, and what would change with turbulent flow, where the heat transfer grows much faster with the speed.